# Placebo Tests

Randomly reassigns treatment to the same number of firms as the real treated group,
then rebuilds the full set of quarterly panels for each draw. Produces 10 × 4 `.dta`
files (new hires, separations, promotions, stock) used by the placebo envelope plot
in `analysis.R`.

**Prerequisites:** run `panel_construction.ipynb` first so that `clean_positions` and
`treated_firms` are available — or simply re-derive them here from the same raw inputs
(this notebook is self-contained and does exactly that).

Run all cells top-to-bottom. Edit only the **Config** cell.

## 1. Imports

In [ ]:
import os
import random
import subprocess
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyreadstat", "-q"])
    import pyreadstat


## 2. Config

In [ ]:
# ── Input paths (same as panel_construction.ipynb) ────────────────────────────
POSITIONS_PATH    = "../data/Positions/positions_stock.csv"
POST1_PATH        = "../data/Firm Level/flagged_postings_gpt_predictions.csv"
POST2_PATH        = "../data/Firm Level/flagged_postings_gpt_predictions_2025.csv"
OCC_EXPOSURE_PATH = "../data/AI Exposure Scores/occ_level.csv"

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = "../data/Positions/"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Panel parameters (must match panel_construction.ipynb) ───────────────────
PANEL_START   = "2021Q1"
PANEL_END     = "2025Q2"
TREAT_Q       = "2022Q4"
HIRE_START    = "2021-01-01"
HIRE_END      = "2025-07-01"
TREATED_START = "2022-11-30"
TREATED_END   = "2025-07-01"
THRESHOLD     = 20

# ── Placebo parameters ────────────────────────────────────────────────────────
N_PLACEBO    = 10      # increase to 50+ for final paper
SEED_OFFSET  = 100     # seeds used: 100, 101, ..., 100 + N_PLACEBO - 1

# ── Industry exclusions ───────────────────────────────────────────────────────
RECRUITMENT_INDUSTRIES = [
    "Recruitment and Staffing Services",
    "Employment and Staffing Services",
    "Employment and Recruitment Services",
    "Human Resources and Recruitment Services",
    "Online Employment Platforms",
    "Human Resources and Workforce Solutions",
    "Business Process Outsourcing Services",
]

STATA_EPOCH = pd.Period("1960Q1", freq="Q")


## 3. Panel Helpers

In [ ]:
def make_firm_quarter_grid(active_firms: set, quarters: pd.PeriodIndex) -> pd.DataFrame:
    return (
        pd.DataFrame({"company_name": sorted(active_firms)}).assign(_tmp=1)
        .merge(pd.DataFrame({"quarter": quarters}).assign(_tmp=1), on="_tmp")
        .drop(columns="_tmp")
    )


def add_common_panel_columns(panel: pd.DataFrame, treated_firms: set,
                              treat_q: str) -> pd.DataFrame:
    out = panel.copy()
    out["treated"]      = out["company_name"].isin(treated_firms).astype(int)
    out["Treatment"]    = out["treated"].map({0: "Control", 1: "Ever Treated"})
    out["post"]         = (out["quarter"] >= pd.Period(treat_q, freq="Q")).astype(int)
    out["quarter_date"] = out["quarter"].dt.to_timestamp()
    out["firm_id"]      = out["company_name"].astype("category").cat.codes + 1
    out["tq"]           = out["quarter"].apply(lambda p: (p - STATA_EPOCH).n).astype(int)
    return out


def add_log_columns(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.copy()
    for c in [col for col in out.columns if col.startswith(prefix)]:
        out[f"log_{c}"] = np.log1p(out[c])
    return out


## 4. Exposure, Transition & Event-Flagging Helpers

In [ ]:
DEFAULT_SENIORITY_GROUPS = {
    "lvl1":   lambda s: s == 1,
    "lvl2":   lambda s: s == 2,
    "lvl3_7": lambda s: s.between(3, 7),
    "lvl1_2": lambda s: s.between(1, 2),
    "lvl2_7": lambda s: s.between(2, 7),
    "lvl1_7": lambda s: s.between(1, 7),
}


def attach_occ_exposure(
    positions: pd.DataFrame,
    occ_exposure_path: str,
    positions_onet_col: str = "onet_code",
    exposure_onet_col: str = "O*NET-SOC Code",
    exposure_value_col: str = "human_rating_beta",
    drop_missing_exposure: bool = False,
) -> tuple:
    occ = pd.read_csv(occ_exposure_path)
    occ[exposure_onet_col]  = occ[exposure_onet_col].astype(str).str.strip()
    occ[exposure_value_col] = pd.to_numeric(occ[exposure_value_col], errors="coerce")

    pos = positions.copy()
    pos[positions_onet_col] = pos[positions_onet_col].astype(str).str.strip()

    merged = pos.merge(
        occ[[exposure_onet_col, exposure_value_col]]
           .drop_duplicates(subset=[exposure_onet_col]),
        left_on=positions_onet_col, right_on=exposure_onet_col, how="left",
    )
    med = merged[exposure_value_col].median(skipna=True)
    merged["high_exposure"] = np.where(
        merged[exposure_value_col].notna(),
        (merged[exposure_value_col] > med).astype(int),
        np.nan,
    )
    if drop_missing_exposure:
        before = len(merged)
        merged = merged.dropna(subset=["high_exposure"]).copy()
        merged["high_exposure"] = merged["high_exposure"].astype(int)
        print(f"  Dropped {before - len(merged):,} rows missing exposure | Remaining: {len(merged):,}")
    print(f"  Exposure column: {exposure_value_col} | Median: {med:.4f}")
    return merged, med


def annotate_worker_transitions(positions: pd.DataFrame) -> pd.DataFrame:
    """Add prev_* and next_* spell columns. Run on FULL cleaned positions."""
    df = positions.sort_values(
        ["user_id", "startdate", "enddate", "company_name"], kind="mergesort"
    ).reset_index(drop=True).copy()

    for shift, prefix in [(1, "prev"), (-1, "next")]:
        for col in ["company_name", "startdate", "enddate", "seniority"]:
            df[f"{prefix}_{col}"] = df.groupby("user_id")[col].shift(shift)
    return df


def flag_events(positions: pd.DataFrame) -> pd.DataFrame:
    """
    is_new_hire   : first observed job OR moved in from another firm
                    (same-firm internal transitions excluded so promotions are not double-counted)
    is_promotion  : spell starts at same firm with strictly higher seniority
    is_separation : spell ends and next spell is elsewhere or missing
    """
    df = positions.copy()
    df["is_new_hire"] = (
        df["prev_company_name"].isna() |
        (df["prev_company_name"] != df["company_name"])
    ).astype(int)

    df["is_promotion"] = (
        df["prev_company_name"].eq(df["company_name"]) &
        df["prev_seniority"].notna() &
        (df["seniority"] > df["prev_seniority"])
    ).astype(int)

    df["is_separation"] = (
        df["enddate"].notna() &
        (
            df["next_company_name"].isna() |
            (df["next_company_name"] != df["company_name"])
        )
    ).astype(int)
    return df


## 5. Quarter-Level Aggregation Helpers

In [ ]:
def _make_seniority_columns(event_df, event_name, seniority_groups,
                             seniority_col="seniority"):
    out = {}
    out[f"{event_name}_total"] = (
        event_df.groupby("company_name")["user_id"].nunique()
        .rename(f"{event_name}_total")
    )
    s = event_df[seniority_col]
    for label, fn in seniority_groups.items():
        out[f"{event_name}_{label}"] = (
            event_df.loc[fn(s)].groupby("company_name")["user_id"].nunique()
            .rename(f"{event_name}_{label}")
        )
    if "high_exposure" in event_df.columns:
        out[f"{event_name}_lowexp"] = (
            event_df.loc[event_df["high_exposure"] == 0]
            .groupby("company_name")["user_id"].nunique()
            .rename(f"{event_name}_lowexp")
        )
        out[f"{event_name}_highexp"] = (
            event_df.loc[event_df["high_exposure"] == 1]
            .groupby("company_name")["user_id"].nunique()
            .rename(f"{event_name}_highexp")
        )
    return out


def _make_stock_columns(stock_df, stock_name, seniority_groups,
                         seniority_col="seniority",
                         exposure_value_col="human_rating_beta"):
    out = {}
    out[f"{stock_name}_total"] = (
        stock_df.groupby("company_name")["user_id"].nunique()
        .rename(f"{stock_name}_total")
    )
    s = stock_df[seniority_col]
    for label, fn in seniority_groups.items():
        out[f"{stock_name}_{label}"] = (
            stock_df.loc[fn(s)].groupby("company_name")["user_id"].nunique()
            .rename(f"{stock_name}_{label}")
        )
    if "high_exposure" in stock_df.columns:
        out[f"{stock_name}_lowexp"] = (
            stock_df.loc[stock_df["high_exposure"] == 0]
            .groupby("company_name")["user_id"].nunique()
            .rename(f"{stock_name}_lowexp")
        )
        out[f"{stock_name}_highexp"] = (
            stock_df.loc[stock_df["high_exposure"] == 1]
            .groupby("company_name")["user_id"].nunique()
            .rename(f"{stock_name}_highexp")
        )
    if exposure_value_col in stock_df.columns:
        out[f"{stock_name}_exposure_mean"]   = (
            stock_df.groupby("company_name")[exposure_value_col].mean()
            .rename(f"{stock_name}_exposure_mean")
        )
        out[f"{stock_name}_exposure_median"] = (
            stock_df.groupby("company_name")[exposure_value_col].median()
            .rename(f"{stock_name}_exposure_median")
        )
    return out


def build_event_panel_quarterly(
    positions, event_name, event_flag_col, event_date_col,
    treated_firms, active_firms, panel_start_q, panel_end_q, treat_q,
    seniority_groups=None, active_threshold=0, out_path=None,
):
    if seniority_groups is None:
        seniority_groups = DEFAULT_SENIORITY_GROUPS

    quarters = pd.period_range(panel_start_q, panel_end_q, freq="Q")
    pos = positions[positions["company_name"].isin(active_firms)].copy()
    pos = pos[pos["seniority"].between(1, 7)].copy()

    pieces = []
    for q in tqdm(quarters, desc=f"{event_name}", leave=False):
        q_s = q.start_time;  q_e = (q + 1).start_time
        event_df = pos[
            (pos[event_flag_col] == 1)
            & pos[event_date_col].notna()
            & (pos[event_date_col] >= q_s)
            & (pos[event_date_col] < q_e)
        ].copy()
        cols = _make_seniority_columns(event_df, event_name, seniority_groups)
        qdf  = pd.DataFrame(index=sorted(active_firms))
        for _, ser in cols.items():
            qdf = qdf.join(ser, how="left")
        qdf = qdf.fillna(0).reset_index().rename(columns={"index": "company_name"})
        qdf["quarter"] = q
        pieces.append(qdf)

    panel = make_firm_quarter_grid(active_firms, quarters).merge(
        pd.concat(pieces, ignore_index=True), on=["company_name", "quarter"], how="left"
    )
    event_cols = [c for c in panel.columns if c.startswith(f"{event_name}_")]
    for c in event_cols:
        panel[c] = panel[c].fillna(0).astype(int)

    panel = add_common_panel_columns(panel, treated_firms, treat_q)
    panel = add_log_columns(panel, prefix=f"{event_name}_")

    stata_cols = (["firm_id", "company_name", "tq", "treated", "post"] +
                  [c for c in panel.columns
                   if c.startswith(f"{event_name}_") or c.startswith(f"log_{event_name}_")])
    stata_df = panel[stata_cols].copy()
    for c in ["firm_id", "tq", "treated", "post"]:
        stata_df[c] = stata_df[c].astype(int)
    for c in stata_df.columns:
        if c not in ["firm_id", "company_name", "tq", "treated", "post"]:
            stata_df[c] = stata_df[c].astype(float)

    if out_path is not None:
        pyreadstat.write_dta(stata_df, out_path, version=14)
    return panel, stata_df


def build_stock_panel_quarterly(
    positions, treated_firms, active_firms,
    panel_start_q, panel_end_q, treat_q,
    seniority_groups=None, active_threshold=0,
    stock_name="employees", exposure_value_col="human_rating_beta",
    out_path=None,
):
    if seniority_groups is None:
        seniority_groups = DEFAULT_SENIORITY_GROUPS

    quarters = pd.period_range(panel_start_q, panel_end_q, freq="Q")
    pos = positions[positions["company_name"].isin(active_firms)].copy()
    pos = pos[pos["seniority"].between(1, 7)].copy()

    pieces = []
    for q in tqdm(quarters, desc="stock", leave=False):
        q_s, q_e = q.start_time, q.end_time
        stock_df = pos[
            pos["startdate"].notna()
            & (pos["startdate"] <= q_e)
            & (pos["enddate"].isna() | (pos["enddate"] >= q_s))
        ].copy()
        cols = _make_stock_columns(stock_df, stock_name, seniority_groups,
                                   exposure_value_col=exposure_value_col)
        qdf  = pd.DataFrame(index=sorted(active_firms))
        for _, ser in cols.items():
            qdf = qdf.join(ser, how="left")
        qdf = qdf.fillna(0).reset_index().rename(columns={"index": "company_name"})
        qdf["quarter"] = q
        pieces.append(qdf)

    panel = make_firm_quarter_grid(active_firms, quarters).merge(
        pd.concat(pieces, ignore_index=True), on=["company_name", "quarter"], how="left"
    )
    stock_cols = [c for c in panel.columns if c.startswith(f"{stock_name}_")]
    for c in stock_cols:
        if c.endswith("_mean") or c.endswith("_median"):
            panel[c] = panel[c].astype(float)
        else:
            panel[c] = panel[c].fillna(0).astype(int)

    panel = add_common_panel_columns(panel, treated_firms, treat_q)
    panel = add_log_columns(panel, prefix=f"{stock_name}_")

    stata_cols = (["firm_id", "company_name", "tq", "treated", "post"] +
                  [c for c in panel.columns
                   if c.startswith(f"{stock_name}_") or c.startswith(f"log_{stock_name}_")])
    stata_df = panel[stata_cols].copy()
    for c in ["firm_id", "tq", "treated", "post"]:
        stata_df[c] = stata_df[c].astype(int)
    for c in stata_df.columns:
        if c not in ["firm_id", "company_name", "tq", "treated", "post"]:
            stata_df[c] = stata_df[c].astype(float)

    if out_path is not None:
        pyreadstat.write_dta(stata_df, out_path, version=14)
    return panel, stata_df


## 6. Master Panel Builder (`build_quarterly_eda_event_panels`)

In [ ]:
def build_quarterly_eda_event_panels(
    positions: pd.DataFrame,
    treated_firms: set,
    active_firms: set,
    occ_exposure_path: str,
    panel_start_q: str,
    panel_end_q: str,
    treat_q: str,
    active_threshold: int = 0,
    seniority_groups: dict = None,
    positions_onet_col: str = "onet_code",
    exposure_onet_col: str = "O*NET-SOC Code",
    exposure_value_col: str = "human_rating_beta",
    onet_filter: list = None,
    out_dir: str = None,
    out_stub: str = "",
) -> dict:
    """
    Full pipeline per treated-firm assignment:
      1. (Optional) subset to O*NET codes
      2. Merge AI exposure scores
      3. Annotate worker transitions (prev/next spell)
      4. Flag new hire / separation / promotion events
      5. Build quarterly firm-level panels for all four outcomes
    """
    if seniority_groups is None:
        seniority_groups = DEFAULT_SENIORITY_GROUPS

    pos_in = positions.copy()

    if onet_filter is not None:
        onet_filter = [str(x).strip() for x in onet_filter]
        pos_in[positions_onet_col] = pos_in[positions_onet_col].astype(str).str.strip()
        pos_in = pos_in[pos_in[positions_onet_col].isin(onet_filter)].copy()
        print(f"  O*NET filter: {len(onet_filter)} codes | Rows: {len(pos_in):,}")

    pos_exp, exposure_median = attach_occ_exposure(
        positions=pos_in,
        occ_exposure_path=occ_exposure_path,
        positions_onet_col=positions_onet_col,
        exposure_onet_col=exposure_onet_col,
        exposure_value_col=exposure_value_col,
        drop_missing_exposure=False,
    )

    pos_trans = annotate_worker_transitions(pos_exp)
    pos_trans = flag_events(pos_trans)

    # Build output paths
    new_hires_out = separations_out = promotions_out = stock_out = None
    if out_dir is not None:
        onet_stub = ""
        if onet_filter is not None:
            safe = [c.replace(".", "").replace("-", "_") for c in onet_filter]
            onet_stub = f"_onet_{safe[0]}" if len(safe) == 1 else "_onet_multi"
        suffix = f"_active_ge{active_threshold}{onet_stub}{out_stub}.dta"
        new_hires_out   = f"{out_dir}firm_quarter_new_hires{suffix}"
        separations_out = f"{out_dir}firm_quarter_separations{suffix}"
        promotions_out  = f"{out_dir}firm_quarter_promotions{suffix}"
        stock_out       = f"{out_dir}firm_quarter_stock{suffix}"

    new_hires_panel, new_hires_stata = build_event_panel_quarterly(
        positions=pos_trans, event_name="new_hires",
        event_flag_col="is_new_hire", event_date_col="startdate",
        treated_firms=treated_firms, active_firms=active_firms,
        panel_start_q=panel_start_q, panel_end_q=panel_end_q, treat_q=treat_q,
        seniority_groups=seniority_groups, active_threshold=active_threshold,
        out_path=new_hires_out,
    )
    separations_panel, separations_stata = build_event_panel_quarterly(
        positions=pos_trans, event_name="separations",
        event_flag_col="is_separation", event_date_col="enddate",
        treated_firms=treated_firms, active_firms=active_firms,
        panel_start_q=panel_start_q, panel_end_q=panel_end_q, treat_q=treat_q,
        seniority_groups=seniority_groups, active_threshold=active_threshold,
        out_path=separations_out,
    )
    promotions_panel, promotions_stata = build_event_panel_quarterly(
        positions=pos_trans, event_name="promotions",
        event_flag_col="is_promotion", event_date_col="startdate",
        treated_firms=treated_firms, active_firms=active_firms,
        panel_start_q=panel_start_q, panel_end_q=panel_end_q, treat_q=treat_q,
        seniority_groups=seniority_groups, active_threshold=active_threshold,
        out_path=promotions_out,
    )
    stock_panel, stock_stata = build_stock_panel_quarterly(
        positions=pos_trans, treated_firms=treated_firms, active_firms=active_firms,
        panel_start_q=panel_start_q, panel_end_q=panel_end_q, treat_q=treat_q,
        seniority_groups=seniority_groups, active_threshold=active_threshold,
        stock_name="employees", exposure_value_col=exposure_value_col,
        out_path=stock_out,
    )

    return {
        "positions_with_exposure_and_flags": pos_trans,
        "exposure_median":    exposure_median,
        "new_hires_panel":    new_hires_panel,
        "separations_panel":  separations_panel,
        "promotions_panel":   promotions_panel,
        "stock_panel":        stock_panel,
        "new_hires_stata":    new_hires_stata,
        "separations_stata":  separations_stata,
        "promotions_stata":   promotions_stata,
        "stock_stata":        stock_stata,
    }


## 7. Load & Clean Positions (Same as `panel_construction.ipynb`)

In [ ]:
def clean_contained_positions(positions: pd.DataFrame) -> pd.DataFrame:
    FAR_FUTURE = pd.Timestamp("2099-12-31")
    df = positions.copy().reset_index(drop=True)
    df["_end_filled"] = df["enddate"].fillna(FAR_FUTURE)
    df["_row"] = df.index

    print("  Step 1/4: Self-joining …")
    keys = df[["_row", "user_id", "company_name", "startdate", "_end_filled", "seniority"]]
    pairs = keys.merge(
        keys.rename(columns={"_row": "_row_j", "startdate": "_start_j",
                              "_end_filled": "_end_j", "seniority": "_sen_j"}),
        on=["user_id", "company_name"],
    )
    print("  Step 2/4: Filtering …")
    contained = pairs[
        (pairs["_row"] != pairs["_row_j"])
        & (pairs["_start_j"] >= pairs["startdate"])
        & (pairs["_end_j"]   <= pairs["_end_filled"])
    ].copy()
    n_pairs = len(contained)
    print(f"  Found {n_pairs:,} containment pairs")
    if n_pairs == 0:
        return df.drop(columns=["_end_filled", "_row"])

    print("  Step 3/4: Classifying …")
    promotions = contained[contained["seniority"] <= contained["_sen_j"]]
    errors     = contained[contained["seniority"] >  contained["_sen_j"]]
    drop_rows  = set(errors["_row_j"].unique())

    shorten = pd.DataFrame(columns=["_row", "_new_end"])
    if len(promotions) > 0:
        shorten = (
            promotions.groupby("_row")["_start_j"].min().reset_index()
            .rename(columns={"_start_j": "_new_end_raw"})
        )
        shorten["_new_end"] = shorten["_new_end_raw"] - pd.Timedelta(days=1)
        shorten = shorten[~shorten["_row"].isin(drop_rows)]

    print("  Step 4/4: Applying edits …")
    rows_to_shorten = []
    if len(shorten) > 0:
        shorten_idx  = shorten.set_index("_row")["_new_end"]
        current_ends = df.loc[shorten_idx.index, "enddate"]
        mask         = shorten_idx < current_ends.fillna(FAR_FUTURE)
        rows_to_shorten = shorten_idx[mask].index.tolist()
        df.loc[rows_to_shorten, "enddate"] = shorten_idx[mask].values

    df = df[~df["_row"].isin(drop_rows)].drop(columns=["_end_filled", "_row"])
    print(f"  Done — shortened: {len(rows_to_shorten):,} | "
          f"dropped: {len(drop_rows):,} | rows remaining: {len(df):,}")
    return df.reset_index(drop=True)


def load_and_clean_positions(positions_path: str,
                             recruitment_industries: list) -> pd.DataFrame:
    print("=" * 60)
    print("LOADING AND CLEANING POSITIONS")
    print("=" * 60)
    pos = pd.read_csv(positions_path)
    pos["startdate"]    = pd.to_datetime(pos["startdate"],    errors="coerce")
    pos["enddate"]      = pd.to_datetime(pos["enddate"],      errors="coerce")
    pos["company_name"] = pos["company_name"].astype(str).str.strip().str.lower()
    pos = pos.dropna(subset=["company_name", "startdate", "user_id"]).copy()
    print(f"Rows loaded: {len(pos):,}")

    for col in ("rics_k400", "seniority"):
        if col not in pos.columns:
            raise ValueError(f"Required column '{col}' not found.")
    pos["seniority"] = pd.to_numeric(pos["seniority"], errors="coerce")
    pos = pos.dropna(subset=["seniority"]).copy()
    pos["seniority"] = pos["seniority"].astype(int)

    bad_firms = pos.loc[pos["rics_k400"].isin(recruitment_industries), "company_name"].unique()
    pos = pos[~pos["company_name"].isin(bad_firms)].copy()
    print(f"After dropping {len(bad_firms):,} recruitment firms: {len(pos):,} rows")

    print("\nResolving contained positions …")
    pos = clean_contained_positions(pos)
    print(f"\nClean positions ready: {len(pos):,} rows")
    return pos.reset_index(drop=True)


def load_treated_firms(post1_path, post2_path,
                       treated_start, treated_end) -> set:
    postings = pd.concat([pd.read_csv(post1_path), pd.read_csv(post2_path)],
                         ignore_index=True)
    postings["post_date"] = pd.to_datetime(postings["post_date"], errors="coerce")
    postings["company"]   = postings["company"].astype(str).str.strip().str.lower()
    postings["is_integrator_gpt"] = (
        pd.to_numeric(postings["is_integrator_gpt"], errors="coerce").fillna(0).astype(int)
    )
    treated = set(
        postings[
            (postings["is_integrator_gpt"] == 1)
            & (postings["post_date"] >= treated_start)
            & (postings["post_date"] <  treated_end)
        ]["company"].unique()
    )
    print(f"Ever-treated firms: {len(treated):,}")
    return treated


def get_active_firms(positions, hire_window_start, hire_window_end,
                     threshold) -> set:
    hs, he = pd.to_datetime(hire_window_start), pd.to_datetime(hire_window_end)
    counts = (
        positions[(positions["startdate"] >= hs) & (positions["startdate"] < he)]
        .groupby("company_name")["user_id"].nunique()
        .reset_index(name="total_hires")
    )
    return set(counts.loc[counts["total_hires"] >= threshold, "company_name"])


### Run: load once

In [ ]:
clean_positions = load_and_clean_positions(POSITIONS_PATH, RECRUITMENT_INDUSTRIES)
treated_firms   = load_treated_firms(POST1_PATH, POST2_PATH, TREATED_START, TREATED_END)
active_firms    = get_active_firms(clean_positions, HIRE_START, HIRE_END, THRESHOLD)

print(f"\nActive firms (>= {THRESHOLD} hires): {len(active_firms):,}")
print(f"Real treated firms:                  {len(treated_firms):,}")
print(f"Treated firms in active set:         "
      f"{len(treated_firms & active_firms):,}")


## 8. Placebo Treatment Assignment

In [ ]:
def generate_placebo_treated_firms(active_firms: set,
                                   real_treated_firms: set,
                                   seed: int = 42) -> set:
    """
    Draw the same number of firms as the real treated group,
    uniformly at random from all active firms (with replacement = False).
    Using random.sample instead of numpy so the seed behaviour exactly matches
    the original code.
    """
    random.seed(seed)
    n_treated  = len(real_treated_firms)
    active_list = list(active_firms)
    return set(random.sample(active_list, n_treated))


## 9. Run Placebo Loop

Each iteration saves 4 `.dta` files:
- `firm_quarter_new_hires_active_ge20_placebo{i}_pretrend2021.dta`
- `firm_quarter_separations_active_ge20_placebo{i}_pretrend2021.dta`
- `firm_quarter_promotions_active_ge20_placebo{i}_pretrend2021.dta`
- `firm_quarter_stock_active_ge20_placebo{i}_pretrend2021.dta`

Note: `analysis.R` reads only the **new hires** placebo files for the placebo envelope
plot, but all four are saved so you can extend the robustness checks later.

In [ ]:
placebo_results = {}

for i in range(N_PLACEBO):
    seed = SEED_OFFSET + i
    placebo_treated = generate_placebo_treated_firms(
        active_firms=active_firms,
        real_treated_firms=treated_firms,
        seed=seed,
    )

    print(f"\n{'='*60}")
    print(f"Placebo {i+1:>2}/{N_PLACEBO}  |  seed={seed}  |  "
          f"placebo treated firms: {len(placebo_treated):,}")
    print(f"{'='*60}")

    results = build_quarterly_eda_event_panels(
        positions=clean_positions,
        treated_firms=placebo_treated,
        active_firms=active_firms,
        occ_exposure_path=OCC_EXPOSURE_PATH,
        panel_start_q=PANEL_START,
        panel_end_q=PANEL_END,
        treat_q=TREAT_Q,
        active_threshold=THRESHOLD,
        out_dir=OUT_DIR,
        out_stub=f"_placebo{i+1}_pretrend2021",
    )

    placebo_results[f"placebo_{i+1}"] = results
    print(f"  ✓ Placebo {i+1} complete.")

print(f"\nDone. {N_PLACEBO} × 4 .dta files written to {OUT_DIR}")


## 10. Sanity Check

In [ ]:
# Verify files exist and spot-check one panel
import glob

written = sorted(glob.glob(f"{OUT_DIR}*_placebo*_pretrend2021.dta"))
print(f"Files written: {len(written)}  (expected {N_PLACEBO * 4})")
for f in written[:8]:
    print(" ", os.path.basename(f))
if len(written) > 8:
    print(f"  ... and {len(written) - 8} more")

# Spot-check: treated share should roughly match real treated share
p1_hires = placebo_results["placebo_1"]["new_hires_stata"]
real_treated_share = len(treated_firms & active_firms) / len(active_firms)
placebo_treated_share = p1_hires["treated"].mean()
print(f"\nReal treated share of active firms : {real_treated_share:.3f}")
print(f"Placebo 1 treated share            : {placebo_treated_share:.3f}  (should match)")
